# Shopkeeper AI — M5 Demand Forecasting Pipeline
End-to-end: load local CSVs → clean → weekly aggregate → feature engineer → train/compare models → save best model.

## 1. Imports

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import joblib

## 2. Load raw CSVs from local `data/` folder

Assumes the M5 CSVs have already been downloaded (e.g. via Kaggle) and placed in a `data/` folder one level above this notebook's working directory. Adjust `project_root` / `base` if your layout differs.

In [ ]:
project_root = Path.cwd().resolve().parent
base = project_root / "data"

calendar = pd.read_csv(base / "calendar.csv")
sales_train_validation = pd.read_csv(base / "sales_train_validation.csv")
sales_train_evaluation = pd.read_csv(base / "sales_train_evaluation.csv")
sell_prices = pd.read_csv(base / "sell_prices.csv")
sample_submission = pd.read_csv(base / "sample_submission.csv")

print("calendar:", calendar.shape)
print("sales_train_validation:", sales_train_validation.shape)
print("sales_train_evaluation:", sales_train_evaluation.shape)
print("sell_prices:", sell_prices.shape)
print("sample_submission:", sample_submission.shape)

## 3. Which sales file to use

`sales_train_validation.csv` covers days `d_1`–`d_1913` (used throughout this pipeline so far). `sales_train_evaluation.csv` covers `d_1`–`d_1941` (28 more days) and is the more complete/up-to-date version — switch `sales_train` to it if you want the extra 4 weeks of history. Sticking with `validation` here to match earlier steps.

In [ ]:
sales_train = sales_train_validation  # swap to sales_train_evaluation for 28 more days of history
sales_train.head()

## 4. Select a larger item subset

Using only 2–3 items gives a near-zero-variance target (confirmed earlier: R² ≈ 0.004). Scaling to ~80 items across one store gives the model enough rows and cross-item variation to learn from.

In [ ]:
STORE_ID = 'CA_1'
N_ITEMS = 80

store_items = sales_train[sales_train['store_id'] == STORE_ID]
sampled_item_ids = store_items['item_id'].drop_duplicates().sample(N_ITEMS, random_state=42).tolist()

subset = sales_train[
    (sales_train['store_id'] == STORE_ID) &
    (sales_train['item_id'].isin(sampled_item_ids))
].copy()

print(subset.shape)

## 5. Melt wide format (d_1 ... d_1913) to long format

In [ ]:
id_cols = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
day_cols = [c for c in subset.columns if c.startswith('d_')]

sales_long = subset.melt(
    id_vars=id_cols,
    value_vars=day_cols,
    var_name='d',
    value_name='sales'
)

print(sales_long.shape)
sales_long.head()

## 6. Merge with calendar to get real dates + wm_yr_wk

In [ ]:
sales_long = sales_long.merge(
    calendar[['d', 'date', 'wm_yr_wk']],
    on='d',
    how='left'
)
sales_long['date'] = pd.to_datetime(sales_long['date'])
sales_long.head()

## 7. Merge with sell_prices

This is where pre-release ("not yet on sale") rows get a NaN price — expected, handled in the next cell.

In [ ]:
sales_df = sales_long.merge(
    sell_prices,
    on=['store_id', 'item_id', 'wm_yr_wk'],
    how='left'
)

print("Rows before cleaning:", len(sales_df))
print("NaN sell_price rows:", sales_df['sell_price'].isna().sum())

## 8. Clean: drop pre-release rows

Confirmed earlier: 100% of `sell_price == 0` rows also had `sales == 0` — these are "item not yet on sale in this store" placeholder rows, not real transactions. Drop them.

In [ ]:
sales_df = sales_df.dropna(subset=['sell_price'])
sales_df = sales_df[sales_df['sell_price'] > 0].copy()

print("Rows after cleaning:", len(sales_df))
print((sales_df['sell_price'] == 0).sum(), "rows with zero price remaining (should be 0)")

## 9. Quick EDA sanity checks

In [ ]:
print(sales_df['sales'].describe())
print(sales_df.groupby('item_id')['date'].agg(['min', 'max', 'count']).head())

daily_total = sales_df.groupby('date')['sales'].sum()
plt.figure(figsize=(14, 4))
daily_total.plot()
plt.title('Total Daily Sales — Sampled Items')
plt.xlabel('Date')
plt.ylabel('Units Sold')
plt.show()

## 10. Aggregate to weekly demand

Daily counts (mostly 0–5) are near-noise for a single item. Weekly totals have real variance and match the business use case (shopkeeper reorders weekly, not daily).

In [ ]:
sales_df['week'] = sales_df['date'].dt.to_period('W').apply(lambda r: r.start_time)

weekly_df = sales_df.groupby(['item_id', 'week']).agg(
    sales=('sales', 'sum'),
    sell_price=('sell_price', 'mean')
).reset_index()

weekly_df = weekly_df.sort_values(['item_id', 'week']).reset_index(drop=True)

print(weekly_df['sales'].describe())

## 11. Feature engineering (weekly)

Lag/rolling features use `shift(1)` before `.rolling()` so no row ever sees its own future value — avoids data leakage.

In [ ]:
for lag in [1, 2, 4, 8]:
    weekly_df[f'lag_{lag}'] = weekly_df.groupby('item_id')['sales'].shift(lag)

for window in [4, 8]:
    weekly_df[f'rolling_mean_{window}'] = (
        weekly_df.groupby('item_id')['sales']
        .shift(1).rolling(window).mean()
        .reset_index(level=0, drop=True)
    )
    weekly_df[f'rolling_std_{window}'] = (
        weekly_df.groupby('item_id')['sales']
        .shift(1).rolling(window).std()
        .reset_index(level=0, drop=True)
    )

weekly_df['month'] = weekly_df['week'].dt.month
weekly_df['week_of_year'] = weekly_df['week'].dt.isocalendar().week.astype(int)
weekly_df['year'] = weekly_df['week'].dt.year

weekly_df['price_change'] = weekly_df.groupby('item_id')['sell_price'].diff()
weekly_df['price_lag_1'] = weekly_df.groupby('item_id')['sell_price'].shift(1)

before = len(weekly_df)
weekly_df = weekly_df.dropna().reset_index(drop=True)
print(f"Dropped {before - len(weekly_df)} rows with NaN lag/rolling features")
print(weekly_df.columns.tolist())

## 12. Time-based train/test split

Never shuffle randomly for time-series data — train on earlier weeks, test on later weeks only.

In [ ]:
categorical_cols = ['item_id']

numeric_cols = [
    'lag_1', 'lag_2', 'lag_4', 'lag_8',
    'rolling_mean_4', 'rolling_std_4',
    'rolling_mean_8', 'rolling_std_8',
    'month', 'week_of_year', 'year',
    'price_change', 'price_lag_1', 'sell_price'
]

target_col = 'sales'

weekly_df = weekly_df.sort_values('week').reset_index(drop=True)
split_date = weekly_df['week'].quantile(0.8)

train_df = weekly_df[weekly_df['week'] < split_date].copy()
test_df  = weekly_df[weekly_df['week'] >= split_date].copy()

X_train = train_df[categorical_cols + numeric_cols]
y_train = train_df[target_col]
X_test  = test_df[categorical_cols + numeric_cols]
y_test  = test_df[target_col]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train range: {train_df['week'].min()} -> {train_df['week'].max()}")
print(f"Test range:  {test_df['week'].min()} -> {test_df['week'].max()}")

## 13. Preprocessing pipeline + baseline check

Always compare against "just predict the mean" — a model that can't beat this baseline isn't learning anything useful.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
    ],
    remainder='passthrough'
)

baseline_pred = [y_train.mean()] * len(y_test)
baseline_mae = mean_absolute_error(y_test, baseline_pred)
print(f"Baseline MAE (predict mean): {baseline_mae:.3f}")
print(f"y_train std: {y_train.std():.3f}  (higher = more learnable signal)")

## 14. Compare models

Evaluated with MAE, RMSE, and R² — R² near 0 means the model isn't beating a naive mean prediction, regardless of how low MAE looks.

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=200, random_state=42, objective='reg:squarederror')
}

results = []
fitted_pipelines = {}

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2})
    fitted_pipelines[name] = pipeline
    print(f"{name:20s} | MAE: {mae:.3f} | RMSE: {rmse:.3f} | R2: {r2:.3f} | Beats baseline: {mae < baseline_mae}")

results_df = pd.DataFrame(results).sort_values('MAE')
results_df

## 15. Save the best model

In [ ]:
best_model_name = results_df.iloc[0]['Model']
best_pipeline = fitted_pipelines[best_model_name]

print(f"Best model: {best_model_name}")

os.makedirs('models', exist_ok=True)
joblib.dump(best_pipeline, 'models/demand_forecast_model.joblib')
print("Saved to models/demand_forecast_model.joblib")